In [1]:
import numpy as np
import astroquery
from astroquery.jplhorizons import Horizons
from astropy.time import Time
from tqdm import tqdm

In [2]:
def astroquery_lookup(id,epoch,id_type='designation',location='@0',ref='earth'):
    #default location is solar system barycenter
    #default reference plane is the equatorial
    #units are AU, and AU/day
    #epoch is UTC for ephmereis queries TDB for element or vector
    spkid = str(2000000+int(id))
    obj = Horizons(id=spkid, id_type=id_type, location=location,epochs=epoch) #default location
    state_vec = np.array(list(np.array(obj.vectors(refplane=ref,cache=False)['x','y','z','vx','vy','vz'])[0]))
    #print(state_vec)
    return state_vec

In [4]:
jd_initial = 2451545.0
data = np.genfromtxt('targets.csv',delimiter=',',dtype=str)
numerical_data = data[1:]
mpc_numbers = numerical_data[:,0].astype(int)

outputs = []
failed_numbers = []
for i in tqdm(range(len(mpc_numbers))):
    mpc_num = mpc_numbers[i]
    try:
        asteroid_initial_state=astroquery_lookup(mpc_num, jd_initial)   #rather than use the states I'll just record them
        row = np.concatenate([[mpc_num],asteroid_initial_state]) #add mpcnum as first entry
        outputs.append(row)
    except:
        print("FAILED ON", mpc_num)
        failed_numbers.append(mpc_num)

100%|█████████████████████████████████████| 6139/6139 [1:03:39<00:00,  1.61it/s]


In [6]:
outputs = np.array(outputs)

In [8]:
np.savetxt('target_initial_conditions.csv',outputs,delimiter=',')